# VitaVision Data Quality and Exploratory Data Analysis

This notebook follows a practical data science workflow for checking whether the final VitaVision dataset is ready for model training.

Main goals:

1. Load the final labeled dataset.
2. Explore the structure and distributions.
3. Check data quality: missing values, duplicates, data types, invalid values, and label consistency.
4. Validate labels against the medical reference ranges used by the project.
5. Summarize whether the dataset is ready for machine learning.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load Dataset

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("vitavision_final_labeled_dataset.csv"),
    Path("data/vitavision_final_labeled_dataset.csv"),
]

data_path = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("Could not find vitavision_final_labeled_dataset.csv. Run this notebook from the project root or data folder.")

df = pd.read_csv(data_path)

print(f"Dataset path: {data_path.resolve()}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

## 3. Basic Structure

In [ ]:
df.info()

In [ ]:
expected_columns = ["SEQN", "Age", "Gender", "Nutrient", "Value", "Label"]

schema_check = pd.DataFrame({
    "column": expected_columns,
    "exists": [col in df.columns for col in expected_columns],
    "dtype": [str(df[col].dtype) if col in df.columns else "missing" for col in expected_columns],
})

schema_check

## 4. Missing Values

In [ ]:
missing_summary = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100).round(3)
missing_summary.sort_values("missing_count", ascending=False)

## 5. Duplicate Checks

In [ ]:
full_duplicate_count = df.duplicated().sum()
key_duplicate_count = df.duplicated(subset=["SEQN", "Nutrient", "Value", "Label"]).sum()

pd.DataFrame({
    "check": ["Full row duplicates", "SEQN + Nutrient + Value + Label duplicates"],
    "count": [full_duplicate_count, key_duplicate_count],
    "percent": [full_duplicate_count / len(df) * 100, key_duplicate_count / len(df) * 100],
}).round(3)

## 6. Value Counts and Distributions

In [ ]:
df["Label"].value_counts(dropna=False).to_frame("count").assign(
    percent=lambda x: (x["count"] / len(df) * 100).round(2)
)

In [ ]:
df["Nutrient"].value_counts(dropna=False).to_frame("count").assign(
    percent=lambda x: (x["count"] / len(df) * 100).round(2)
)

In [ ]:
pd.crosstab(df["Nutrient"], df["Label"], margins=True)

## 7. Numeric Feature Summary

In [ ]:
numeric_columns = ["Age", "Gender", "Value"]

df[numeric_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
df.groupby("Nutrient")["Value"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(3)

## 8. Invalid Value Checks

In [ ]:
invalid_checks = pd.Series({
    "age_below_0": (df["Age"] < 0).sum(),
    "age_above_120": (df["Age"] > 120).sum(),
    "gender_not_1_or_2": (~df["Gender"].isin([1, 2])).sum(),
    "value_less_or_equal_0": (df["Value"] <= 0).sum(),
    "label_not_expected": (~df["Label"].isin(["Deficient", "Normal", "Excessive"])).sum(),
})

invalid_checks.to_frame("count")

## 9. IQR Outlier Review

This is an exploratory check only. Medical lab data can naturally contain extreme values, so outliers should be reviewed before removal.

In [ ]:
def iqr_outlier_summary(group: pd.DataFrame) -> pd.Series:
    q1 = group["Value"].quantile(0.25)
    q3 = group["Value"].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (group["Value"] < lower) | (group["Value"] > upper)
    return pd.Series({
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": mask.sum(),
        "outlier_percent": mask.mean() * 100,
    })

outlier_summary = df.groupby("Nutrient", group_keys=False).apply(iqr_outlier_summary)
outlier_summary.round(3)

## 10. Reference Range Label Validation

This section recalculates labels from the project reference ranges and compares them with the dataset labels.

In [ ]:
REFERENCE_RANGES = {
    "Zinc": {"low": 66, "high": 106},
    "Vitamin E": {"low": 500, "high": 2000},
    "Vitamin A": {"low": 28, "high": 86},
    "Vitamin D": {"low": 20, "high": 50},
    "Vitamin C": {"low": 0.4, "high": 2.0},
    "Magnesium": {"low": 1.7, "high": 2.2},
    "Folate": {"low": 3, "high": 20},
    "Vitamin K": {"low": 0.10, "high": 2.20},
    "Vitamin B12": {"low": 200, "high": 900},
    "Vitamin B6": {"low": 20, "high": 100},
    "Calcium": {"low": 8.6, "high": 10.2},
}

def expected_label(row):
    nutrient = row["Nutrient"]
    value = row["Value"]
    gender = row["Gender"]

    if pd.isna(value):
        return np.nan

    if nutrient == "Ferritin":
        if gender == 1:
            low, high = 30, 400
        else:
            low, high = 13, 150
    elif nutrient in REFERENCE_RANGES:
        low = REFERENCE_RANGES[nutrient]["low"]
        high = REFERENCE_RANGES[nutrient]["high"]
    else:
        return "Unknown Nutrient"

    if value < low:
        return "Deficient"
    if value <= high:
        return "Normal"
    return "Excessive"

validation_df = df.copy()
validation_df["Expected_Label"] = validation_df.apply(expected_label, axis=1)
validation_df["Label_Match"] = validation_df["Label"] == validation_df["Expected_Label"]

label_match_summary = validation_df["Label_Match"].value_counts(dropna=False).to_frame("count")
label_match_summary["percent"] = (label_match_summary["count"] / len(validation_df) * 100).round(4)
label_match_summary

In [ ]:
mismatches = validation_df.loc[~validation_df["Label_Match"], [
    "SEQN", "Age", "Gender", "Nutrient", "Value", "Label", "Expected_Label"
]]

print(f"Mismatched labels: {len(mismatches):,}")
mismatches.head(20)

## 11. Patient-Level Split Risk

Because one person can have multiple nutrient rows, a random row-level split can place the same patient in both train and test. A patient-level split using `SEQN` is stronger for evaluation.

In [ ]:
patient_summary = pd.Series({
    "rows": len(df),
    "unique_patients_seqn": df["SEQN"].nunique(),
    "avg_rows_per_patient": len(df) / df["SEQN"].nunique(),
    "max_rows_per_patient": df.groupby("SEQN").size().max(),
})

patient_summary.to_frame("value")

## 12. Training Readiness Summary

In [ ]:
readiness_checks = {
    "required_columns_exist": all(col in df.columns for col in expected_columns),
    "no_missing_values": df[expected_columns].isna().sum().sum() == 0,
    "valid_labels_only": df["Label"].isin(["Deficient", "Normal", "Excessive"]).all(),
    "valid_gender_values": df["Gender"].isin([1, 2]).all(),
    "valid_age_range": df["Age"].between(0, 120).all(),
    "positive_lab_values": (df["Value"] > 0).all(),
    "labels_match_reference_ranges": validation_df["Label_Match"].all(),
}

readiness = pd.DataFrame(
    [{"check": check, "passed": passed} for check, passed in readiness_checks.items()]
)

readiness

## 13. Notes for the Modeling Stage

- Use `Age`, `Gender`, `Nutrient`, and `Value` as model inputs.
- Use `Label` as the target.
- Encode `Nutrient` using one-hot encoding.
- Consider a patient-level train/test split by `SEQN` to avoid patient leakage.
- Report Accuracy, Macro F1-score, classification report, and confusion matrix.
- Explain that labels are medically grounded because they were generated from reference ranges.